In [1]:
import cv2
import numpy as np
import os

# =====================================================
# VIDEO SETTINGS
# =====================================================

VIDEO_PATH = r"C:/Users/Asanthi Upeksha/Desktop/Segmentation_part/vid_1.MOV"

# Main Output Folder
MAIN_OUTPUT_FOLDER = r"C:/Users/Asanthi Upeksha/Desktop/Segmentation2_Output"

# =====================================================
# CREATE OUTPUT FOLDERS
# =====================================================

BEFORE_FOLDER = os.path.join(
    MAIN_OUTPUT_FOLDER,
    "Before_Segmentation"
)

AFTER_FOLDER = os.path.join(
    MAIN_OUTPUT_FOLDER,
    "After_Segmentation"
)

MASK_FOLDER = os.path.join(
    MAIN_OUTPUT_FOLDER,
    "Adaptive_Threshold_Mask"
)

# Create folders
os.makedirs(BEFORE_FOLDER, exist_ok=True)

os.makedirs(AFTER_FOLDER, exist_ok=True)

os.makedirs(MASK_FOLDER, exist_ok=True)


# =====================================================
# IMAGE ENHANCEMENT
# =====================================================

def enhance_frame(frame):

    # Convert to grayscale
    gray = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2GRAY
    )

    # Gaussian Blur
    blur = cv2.GaussianBlur(
        gray,
        (5, 5),
        0
    )

    # Contrast Enhancement
    enhanced = cv2.equalizeHist(blur)

    return enhanced


# =====================================================
# ADAPTIVE THRESHOLD SEGMENTATION
# =====================================================

def adaptive_crack_segmentation(image):

    adaptive_thresh = cv2.adaptiveThreshold(
        image,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        11,
        2
    )

    # Morphological Operations
    kernel = np.ones((3, 3), np.uint8)

    cleaned = cv2.morphologyEx(
        adaptive_thresh,
        cv2.MORPH_OPEN,
        kernel,
        iterations=1
    )

    return cleaned


# =====================================================
# DRAW CRACK CONTOURS
# =====================================================

def draw_crack_contours(frame, mask):

    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    output = frame.copy()

    for cnt in contours:

        area = cv2.contourArea(cnt)

        # Small crack filtering
        if area > 100:

            cv2.drawContours(
                output,
                [cnt],
                -1,
                (0, 255, 0),
                2
            )

    return output


# =====================================================
# MAIN PROCESS
# =====================================================

def process_video():

    cap = cv2.VideoCapture(VIDEO_PATH)

    if not cap.isOpened():

        print("Error opening video")
        return

    frame_count = 0

    print("Starting crack segmentation...")
    print("Press Q to quit")

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        # =================================================
        # BEFORE SEGMENTATION
        # =================================================

        before_segmentation = frame.copy()

        # =================================================
        # IMAGE ENHANCEMENT
        # =================================================

        enhanced = enhance_frame(frame)

        # =================================================
        # ADAPTIVE THRESHOLD SEGMENTATION
        # =================================================

        mask = adaptive_crack_segmentation(
            enhanced
        )

        # =================================================
        # AFTER SEGMENTATION
        # =================================================

        after_segmentation = draw_crack_contours(
            frame,
            mask
        )

        # =================================================
        # SAVE OUTPUTS
        # =================================================

        # Before Segmentation Image
        cv2.imwrite(
            os.path.join(
                BEFORE_FOLDER,
                f"before_{frame_count}.jpg"
            ),
            before_segmentation
        )

        # After Segmentation Image
        cv2.imwrite(
            os.path.join(
                AFTER_FOLDER,
                f"after_{frame_count}.jpg"
            ),
            after_segmentation
        )

        # Adaptive Threshold Mask
        cv2.imwrite(
            os.path.join(
                MASK_FOLDER,
                f"mask_{frame_count}.jpg"
            ),
            mask
        )

        # =================================================
        # DISPLAY WINDOWS
        # =================================================

        cv2.imshow(
            "Before Segmentation",
            before_segmentation
        )

        cv2.imshow(
            "Adaptive Threshold Mask",
            mask
        )

        cv2.imshow(
            "After Segmentation",
            after_segmentation
        )

        frame_count += 1

        # Press q to quit
        key = cv2.waitKey(1) & 0xFF

        if key == ord('q'):
            break

    # =====================================================
    # RELEASE
    # =====================================================

    cap.release()

    cv2.destroyAllWindows()

    print("\nProcessing Completed")
    print("Frames Processed:", frame_count)

    print("\nOutputs saved inside:")
    print(MAIN_OUTPUT_FOLDER)


# =====================================================
# RUN PROGRAM
# =====================================================

if __name__ == "__main__":

    process_video()

Starting crack segmentation...
Press Q to quit

Processing Completed
Frames Processed: 12

Outputs saved inside:
C:/Users/Asanthi Upeksha/Desktop/Segmentation2_Output


In [ ]:
qq